# Lesson 10 - প্রোডাকশনে AI এজেন্টস

এই পাঠে আপনি Microsoft Agent Framework (Python) ব্যবহার করে AI এজেন্টদের জন্য **প্রোডাকশন প্যাটার্ন** শিখবেন। আমরা আচ্ছাদিত করব:

- **অবজারভেবিলিটি** — এজেন্ট ইন্টারঅ্যাকশনে টাইমিং এবং লগিং যোগ করা
- **ইভ্যালুয়েশন** — একটি ইভ্যালুয়েটর এজেন্ট ব্যবহার করে রেসপন্সের গুণগত মান স্কোর করা
- **কস্ট ম্যানেজমেন্ট** — টোকেন অপ্টিমাইজেশন এবং মডেল নির্বাচনের কৌশল

অনুভবটি একটি **ট্রাভেল এজেন্ট** যা ব্যবহারকারীদের ট্রিপ পরিকল্পনায় সাহায্য করে, যার উপর পর্যবেক্ষণ এবং মূল্যায়ন স্তর সংযুক্ত।


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## উৎপাদন বিবেচনা

প্রোটোটাইপ থেকে AI এজেন্টদের উৎপাদনে নিয়ে আসার জন্য তিনটি স্তম্ভের প্রতি সতর্ক মনোযোগ প্রয়োজন:

1. **দর্শনযোগ্যতা** — এজেন্ট কী করছে, কতক্ষণ সময় নিচ্ছে এবং কোন টুলগুলি কল করছে তা বোঝার দৃষ্টিভঙ্গি প্রয়োজন। ট্রেসিং এবং লগিং ছাড়া, উৎপাদনের সমস্যাগুলি ডিবাগ করা প্রায় অসম্ভব।

2. **মূল্যায়ন** — স্বয়ংক্রিয় মান যাচাইকরণ নিশ্চিত করে যে এজেন্টের প্রতিউত্তর সময়ের সঙ্গে সঙ্গেই সঠিক, পূর্ণাঙ্গ এবং সহায়ক থাকে। একটি মূল্যায়নকারী এজেন্ট পূর্বনির্ধারিত মানদণ্ডের বিরুদ্ধে প্রতিউত্তরগুলিকে স্কোর করতে পারে।

3. **ব্যয় ব্যবস্থাপনা** — টোকেন ব্যবহারের সরাসরি প্রভাব পড়ে ব্যয়ের উপর। প্রম্পট অপ্টিমাইজেশন, মডেল নির্বাচন এবং ক্যাশিংয়ের মতো কৌশলগুলি মান বজায় রেখে খরচ নিয়ন্ত্রণে সাহায্য করে।


## একটি অবজারভেবল এজেন্ট তৈরি করা

আমরা ট্রাভেল টুলগুলি সংজ্ঞায়িত করি এবং লেটেন্সি পর্যবেক্ষণের জন্য এজেন্ট কলটি টাইমিং সহ মোড়ানো করি। প্রোডাকশনে আপনি OpenTelemetry বা অনুরূপ ট্রেসিং ব্যাকএন্ডের সাথে একত্রিত করবেন।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## মূল্যায়ন প্যাটার্নস

একটি সাধারণ প্রোডাকশন প্যাটার্ন হল দ্বিতীয় একজন এজেন্টকে **মূল্যায়নকারী** হিসাবে ব্যবহার করা। মূল্যায়নকারী প্রাথমিক এজেন্টের প্রতিক্রিয়াকে পূর্বনির্ধারিত মানদণ্ড যেমন সম্পূর্ণতা, নির্ভুলতা, এবং সহায়কতার বিরুদ্ধে স্কোর করে।

এটি সক্ষম করে:
- ব্যবহারকারীদের কাছে প্রতিক্রিয়া পৌঁছানোর আগে স্বয়ংক্রিয় গুণগত মান গেট
- প্রম্পট বা মডেল পরিবর্তনের সময় রিগ্রেশন সনাক্তকরণ
- সময়ের সাথে এজেন্টের পারফরম্যান্সের ধারাবাহিক মনিটরিং


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## খরচ ব্যবস্থাপনা কৌশলসমূহ

উৎপাদন AI এজেন্টের জন্য খরচ নিয়ন্ত্রণ করা অত্যন্ত গুরুত্বপূর্ণ। এখানে প্রধান কয়েকটি কৌশল দেওয়া হলো:

| কৌশল | বর্ণনা |
|---|---|
| **প্রাম্পট অপ্টিমাইজেশন** | সিস্টেম নির্দেশাবলী সংক্ষিপ্ত রাখুন। ইনপুট টোকেন কমাতে অপ্রয়োজনীয় প্রসঙ্গ অপসারণ করুন। |
| **মডেল নির্বাচন** | শ্রেণীবিভাজন বা নির্যাসের মতো সহজ কাজের জন্য ছোট ও সস্তা মডেল (যেমন GPT-4o-mini) ব্যবহার করুন এবং জটিল বিশ্লেষণের জন্য বড় মডেল সংরক্ষণ করুন। |
| **ক্যাশিং** | টুল ফলাফল এবং প্রায়ই জিজ্ঞাসিত প্রশ্ন ক্যাশ করুন যাতে অপ্রয়োজনীয় API কল এড়ানো যায়। |
| **টোকেন বাজেট** | অপ্রত্যাশিত দীর্ঘ প্রতিক্রিয়া প্রতিরোধ করতে `max_tokens` সীমা নির্ধারণ করুন। |
| **ব্যাচিং** | যেখানে সম্ভব, একাধিক ব্যবহারকারীর প্রশ্ন একক API কল-এ গুছিয়ে পাঠান। |

বাস্তবে, একটি স্তরযুক্ত পদ্ধতি ভাল কাজ করে: সরল অনুরোধগুলি দ্রুত, সাশ্রয়ী মডেলের কাছে পাঠিয়ে দিন এবং শুধুমাত্র জটিল প্রশ্নগুলো আরও সক্ষম মডেলের কাছে বৃদ্ধি করুন।


## সারাংশ

এই পাঠে আপনি শিখেছেন কীভাবে:

1. **এজেন্টের ইন্টারঅ্যাকশনে পর্যবেক্ষণ যোগ করবেন** সময় এবং লগিংসহ, যা ট্রেসিং এবং মনিটরিংয়ের জন্য ভিত্তি স্থাপন করে।
2. **এজেন্টের প্রতিক্রিয়া স্বয়ংক্রিয়ভাবে মূল্যায়ন করবেন** একটি ইভ্যালুয়েটর এজেন্ট ব্যবহার করে যা পূর্ণতা, নির্ভুলতা, এবং সহায়তার স্কোর দেয়।
3. **ব্যয় পরিচালনা করবেন** প্রম্পট অপ্টিমাইজেশন, মডেল নির্বাচনী, ক্যাশিং, এবং টোকেন বাজেটের মাধ্যমে।

এই প্রোডাকশন প্যাটার্নগুলি আপনার AI এজেন্টকে স্কেলে নির্ভরযোগ্য, পরিমাপযোগ্য, এবং ব্যয়-সাশ্রয়ী রাখতে সাহায্য করে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
